# Orca-FA GGUF Remote Inference Server (Google Colab T4 GPU)

Chạy `orca-fa-qwen2.5-3b-q4_k_m.gguf` trực tiếp trên GPU T4 của Google Colab (0% RAM/CPU máy cá nhân, không cần Ollama) và xuất OpenAI-compatible URL để dán vào `LOCAL_GGUF_BASE_URL` trong `.env.local`.

In [ ]:
from pathlib import Path
import json, subprocess, urllib.request, zipfile

GGUF_PATH = Path("/content/orca-fa-qwen2.5-3b-q4_k_m.gguf")
if not GGUF_PATH.exists():
    from google.colab import files
    print("Upload orca-fa-qwen2.5-3b-q4_k_m.gguf:")
    uploaded = files.upload()
    name = next(iter(uploaded.keys()))
    if name != GGUF_PATH.name:
        Path(name).rename(GGUF_PATH)

req = urllib.request.Request(
    "https://api.github.com/repos/ggml-org/llama.cpp/releases/latest",
    headers={"User-Agent": "orca-fa-colab-serve"},
)
rel = json.loads(urllib.request.urlopen(req).read().decode())
asset = next(
    a for a in rel["assets"]
    if "ubuntu" in a["name"].lower() and "x64" in a["name"].lower() and a["name"].endswith((".tar.gz", ".zip")) and "vulkan" not in a["name"].lower()
)
pkg = Path("/content") / asset["name"]
if not Path("/content/llama-bin").exists():
    urllib.request.urlretrieve(asset["browser_download_url"], pkg)
    Path("/content/llama-bin").mkdir(exist_ok=True)
    if pkg.suffix == ".zip":
        with zipfile.ZipFile(pkg) as zf:
            zf.extractall("/content/llama-bin")
    else:
        subprocess.run(["tar", "-xzf", str(pkg), "-C", "/content/llama-bin"], check=True)

if not Path("/usr/local/bin/cloudflared").exists():
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "/usr/local/bin/cloudflared"], check=True)
    Path("/usr/local/bin/cloudflared").chmod(0o755)
print("Ready:", GGUF_PATH)

In [ ]:
import os, re, subprocess, time
from pathlib import Path

server_bin = next(Path("/content/llama-bin").rglob("llama-server"))
server_bin.chmod(0o755)
env = {**os.environ, "LD_LIBRARY_PATH": str(server_bin.parent) + ":" + os.environ.get("LD_LIBRARY_PATH", "")}

srv = subprocess.Popen([
    str(server_bin),
    "-m", "/content/orca-fa-qwen2.5-3b-q4_k_m.gguf",
    "--alias", "orca-fa-qwen2.5-3b",
    "--host", "127.0.0.1", "--port", "8080",
    "-c", "4096", "-ngl", "99",
], env=env, stdout=open("/content/llama_server.log", "w"), stderr=subprocess.STDOUT)

time.sleep(5)
cf = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8080"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for line in cf.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        print(f"\n=== DAN VAO .env.local TREN MAY BAN ===")
        print(f"LOCAL_GGUF_BASE_URL={url}/v1")
        print("LOCAL_GGUF_MODEL=orca-fa-qwen2.5-3b")
        print("LOCAL_GGUF_PRIORITY=primary")
        break